# Day 1 — Multilingual Drift Evaluation

# 1. Goal
Measure how consistent an LLM is across languages when given semantically equivalent prompts.

We evaluate:
- Cross-language drift (EN, DE, PT)
- Effect of stochastic generation (temperature)

Metric:
- Bounded intent drift ∈ [0, 1]

# 2. Imports + Setup

In [15]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
from sentence_transformers import SentenceTransformer

from src.llm.ollama_client import OllamaClient
from src.utils.config import PROMPTS
from src.evaluation.metrics import bounded_intent_drift

# 3. Load Model

In [16]:
llm = OllamaClient(model="mistral:latest")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts):
    return embedder.encode(texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# 4. Load Prompts

In [13]:
en_prompt = PROMPTS["en"]["question"]
de_prompt = PROMPTS["de"]["question"]
pt_prompt = PROMPTS["pt"]["question"]

# 5. Generation function (simple + stable)

In [17]:
def run_prompt(prompt, n=5, temperature=0.7):
    return [
        llm.generate(prompt, temperature=temperature)
        for _ in range(n)
    ]

# 6. Run experiment

In [18]:
en_outputs = run_prompt(en_prompt)
de_outputs = run_prompt(de_prompt)
pt_outputs = run_prompt(pt_prompt)

[Ollama] 39.54s | temp=0.7
[Ollama] 12.62s | temp=0.7
[Ollama] 10.21s | temp=0.7
[Ollama] 13.08s | temp=0.7
[Ollama] 9.26s | temp=0.7
[Ollama] 16.16s | temp=0.7
[Ollama] 14.68s | temp=0.7
[Ollama] 16.84s | temp=0.7
[Ollama] 17.53s | temp=0.7
[Ollama] 18.48s | temp=0.7
[Ollama] 20.18s | temp=0.7
[Ollama] 15.04s | temp=0.7
[Ollama] 13.36s | temp=0.7
[Ollama] 11.73s | temp=0.7
[Ollama] 13.85s | temp=0.7


# 7. Compute Drift

In [21]:
drift = bounded_intent_drift(
    en_outputs,
    de_outputs,
    pt_outputs,
    embed_fn=embed_texts
)

print("Multilingual drift [0–1]:", drift)

Multilingual drift [0–1]: 0.6253443956375122


# 8. Temperature comparison

In [22]:
temps = [0.0, 0.7]
results = {}

for t in temps:
    en = run_prompt(en_prompt, temperature=t)
    de = run_prompt(de_prompt, temperature=t)
    pt = run_prompt(pt_prompt, temperature=t)

    results[t] = bounded_intent_drift(en, de, pt, embed_fn=embed_texts)

results

[Ollama] 13.79s | temp=0.0
[Ollama] 12.21s | temp=0.0
[Ollama] 11.98s | temp=0.0
[Ollama] 11.92s | temp=0.0
[Ollama] 12.44s | temp=0.0
[Ollama] 16.08s | temp=0.0
[Ollama] 13.08s | temp=0.0
[Ollama] 15.02s | temp=0.0
[Ollama] 17.08s | temp=0.0
[Ollama] 16.44s | temp=0.0
[Ollama] 19.44s | temp=0.0
[Ollama] 16.10s | temp=0.0
[Ollama] 16.02s | temp=0.0
[Ollama] 13.76s | temp=0.0
[Ollama] 12.59s | temp=0.0
[Ollama] 14.16s | temp=0.7
[Ollama] 13.39s | temp=0.7
[Ollama] 15.91s | temp=0.7
[Ollama] 16.03s | temp=0.7
[Ollama] 14.93s | temp=0.7
[Ollama] 15.68s | temp=0.7
[Ollama] 12.38s | temp=0.7
[Ollama] 12.86s | temp=0.7
[Ollama] 14.81s | temp=0.7
[Ollama] 16.14s | temp=0.7
[Ollama] 13.84s | temp=0.7
[Ollama] 13.88s | temp=0.7
[Ollama] 347.18s | temp=0.7
[Ollama] 12.42s | temp=0.7
[Ollama] 12.71s | temp=0.7


{0.0: 0.5051435828208923, 0.7: 0.6204155683517456}

# 9. Final Results

## Results

Temperature = 0.0  
→ Drift ≈ 0.50  

Temperature = 0.7  
→ Drift ≈ 0.62  

### Observations
- The model shows moderate multilingual drift even under deterministic decoding
- Increasing temperature leads to higher drift
- This suggests that randomness amplifies cross-lingual inconsistencies

# 9. Final Conclusion

## Conclusion

The evaluated model (mistral:latest) exhibits consistent multilingual drift, indicating that semantically equivalent prompts in different languages produce non-equivalent outputs.

Even at temperature 0.0, drift remains significant (~0.51), suggesting that inconsistencies are inherent to the model rather than purely due to stochastic decoding.

Higher temperature (0.7) further increases drift (~0.62), confirming that randomness exacerbates cross-lingual variability.

These findings highlight the importance of controlling generation parameters when evaluating multilingual robustness in LLMs.